In [23]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.tools import tool

load_dotenv()


def getWeather(location: str) -> str:
    """获取指定城市的当前天气信息。

    Args:
        location: 城市名称，例如 "Beijing"
    """
    return f"Current weather in {location} is sunny"


agent = create_agent(
    "deepseek-chat",
    tools=[getWeather]
)
response = agent.invoke({
    "messages":[
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎个。"),
        AIMessage("你好，虎哥，很高兴认识你。"),
        HumanMessage("北京今天天气如何？")
    ]
})
print(response)

SyntaxError: invalid character '。' (U+3002) (2414605566.py, line 10)

In [11]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是虎个。
================================== Ai Message ==================================

你好，虎哥，很高兴认识你。
================================ Human Message =================================

北京今天天气如何？
================================== Ai Message ==================================

我来帮你查一下北京的天气。
Tool Calls:
  getWeather (call_00_1jE4zMV1R85hSKUsYQQp2888)
 Call ID: call_00_1jE4zMV1R85hSKUsYQQp2888
  Args:
    location: Beijing
================================= Tool Message =================================
Name: getWeather

Current weather in Beijing is sunny
================================== Ai Message ==================================

北京今天天气是**晴天**☀️，适合出门活动。需要我帮你查别的城市吗，虎哥？


In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.tools import tool

load_dotenv()
import os
model = init_chat_model(
    model="qwen3.5-omni-plus",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [3]:
agent = create_agent(model = model)

In [14]:
message = HumanMessage([
    {"type":"text", "text":"描述一下这张照片的内容"},
    {"type":"image","url":"https://ts3.tc.mm.bing.net/th/id/OIP-C.IuNuzsR9jKSqu2-VTCmNOAHaE8?r=0&rs=1&pid=ImgDetMain&o=7&rm=3"}
])

In [15]:
stream = agent.stream(
    {"messages":[message]},
    stream_mode = "messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)


这张照片是一幅充满温暖与自然气息的特写作品，主体是一朵盛开的向日葵。

**画面内容描述：**

- **主体花朵**：一朵金黄色的向日葵占据画面中心偏左的位置，花瓣层层叠叠、舒展饱满，边缘略带柔和的弧度，呈现出阳光般的暖色调。花盘部分呈深褐色至橄榄绿色，布满细密的小花蕊，纹理清晰可见，展现出生命的细节与质感。
  
- **背景环境**：背景是虚化的绿色田野或草地，营造出浅景深效果，使焦点完全集中在向日葵上。远处隐约可见模糊的树影或地平线，天空呈现淡黄色调，暗示着可能是清晨或傍晚时分的柔和光线。

- **光影氛围**：整张照片被温暖的金色光晕笼罩，仿佛沐浴在夕阳余晖中，给人一种宁静、治愈、充满希望的感觉。光线从侧后方照射，在花瓣边缘形成柔和高光，增强了立体感和通透感。

- **构图与风格**：采用近景特写构图，突出花朵的自然之美；色彩以黄、绿为主，搭配柔和的暖调背景，整体风格清新、诗意，富有艺术感染力。

**总结：**
这是一张展现自然之美的摄影佳作，通过聚焦一朵向日葵，在柔和的光线与朦胧的背景衬托下，传递出生命力、温暖与宁静的意境，令人感受到大自然的温柔与力量。

In [29]:
from ipywidgets import FileUpload, Output
from IPython.display import display, Image

uploader = FileUpload(accept='image/*', multiple=False)
out = Output()
display(uploader, out)

def on_upload(change):
    print("回调触发了")
    with out:
        out.clear_output()
        if not uploader.value:
            print("没有文件")
            return
        info = uploader.value[0]
        print("文件名:", info['name'])
        print("大小:", info['size'], "字节")
        display(Image(data=info['content']))
        print("上传成功！")

uploader.observe(on_upload, names='value')
"""from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept="*", multiple=False)
display(uploader)"""

FileUpload(value=(), accept='image/*', description='Upload')

Output()

'from ipywidgets import FileUpload\nfrom IPython.display import display\n\nuploader = FileUpload(accept="*", multiple=False)\ndisplay(uploader)'

In [27]:
print(uploader.value)

()


In [6]:
import base64

uploader_file = uploader.value[0]
content_mv = uploader_file["content"]
img_bytes = bytes(content_mv)
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

IndexError: tuple index out of range

In [7]:
mutimodal_question = HumanMessage(content=[
    {
        "type":"image",
        "base64":img_b64,
        "mime_type":"image/png",
    },
    {"type":"text","text": "给我讲讲图片中的城市"}
])
for chunk, metadata in agent.stream(
        {"messages":[mutimodal_question]},
        stream_mode = "messages"
):
    print(chunk.content, end="", flush=True)

NameError: name 'img_b64' is not defined

In [16]:
import base64

with open(r"D:\data\AI agent\photo.png", "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("utf-8")

print("OK，长度：", len(img_b64))

OK，长度： 408332


In [17]:
from langchain_core.messages import HumanMessage

message = HumanMessage([
    {"type": "text", "text": "描述一下这张照片的内容"},
    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
])

stream = agent.stream({"messages": [message]}, stream_mode="messages")
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

这张照片是一张标准的证件照或正式肖像照，内容如下：

- **人物主体**：一位年轻的东亚男性，面容清秀，皮肤白皙，五官端正。他正视镜头，表情自然、温和，嘴角微微上扬，带着一丝浅笑。
- **发型与妆容**：黑色短发，梳理整齐，略带蓬松感；眉毛修饰得清晰有型，眼部干净明亮，整体妆感淡雅自然（可能经过轻微修图或化妆）。
- **服装搭配**：
  - 外穿一件浅米色或灰白色的西装外套，剪裁合体，显得干练优雅；
  - 内搭一件纯白色衬衫，领口挺括；
  - 佩戴一个经典的黑色蝴蝶结领结（Bow Tie），增添正式感和时尚气息。
- **背景**：纯色亮蓝色背景，常见于官方证件照或职业形象照，突出人物主体，视觉效果简洁明快。
- **光线与构图**：正面打光均匀柔和，无明显阴影；构图为胸部以上特写，居中对称，符合标准人像摄影规范。

✅ 总体风格：专业、整洁、青春、得体，适合用于简历、学生证、护照、公司官网等正式场合。

---  
这是一张精心拍摄并可能经过后期处理的现代青年正式肖像照，展现出自信、稳重又不失亲和力的形象。